# Business Entity Resolution — Learning Project

Goal: learn ML/DS fundamentals end-to-end by solving a real Entity Resolution problem.

**Pipeline overview:**
1. Load & Explore
2. Understand the "messiness" (name/address noise)
3. Blocking / Candidate Generation
4. Feature Engineering (string similarity)
5. Model Training (binary classifier: match / not match)
6. Validation Split + F0.5 Evaluation
7. Generate `matching_results.tsv` and `candidate_pairs.tsv` for test set

We'll fill in each section together, one at a time. Run cells top to bottom as we go;
don't skip ahead to sections marked TODO until we get there.


## Step 1: Load & Explore

Goal: get a feel for the data before writing a single line of ML code.

In [1]:
import pandas as pd

# sep="\t" is critical here -- these are TSV files, not CSV.
# If you forget this, pandas will silently load the whole line as ONE column
# and everything downstream will look broken in confusing ways.
s1 = pd.read_csv("dataset/train/train_source1.tsv", sep="\t")
s2 = pd.read_csv("dataset/train/train_source2.tsv", sep="\t")
s3 = pd.read_csv("dataset/train/train_source3.tsv", sep="\t")
gt = pd.read_csv("dataset/train/train_ground_truth.tsv", sep="\t")


### 1.1 Basic shape
`.shape` returns `(num_rows, num_columns)` — tells you scale at a glance.

In [2]:
print("Source 1 shape:", s1.shape)
print("Source 2 shape:", s2.shape)
print("Source 3 shape:", s3.shape)
print("Ground truth shape:", gt.shape)


Source 1 shape: (2206821, 4)
Source 2 shape: (5034616, 4)
Source 3 shape: (5285603, 4)
Ground truth shape: (2206821, 2)


### 1.2 Peek at the rows
`.head()` shows the first 5 rows — sanity check that the tab-separated read worked (you should see 4 distinct columns, not 1 giant one).

In [3]:
s1.head()

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [4]:
gt.head()

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


### 1.3 Data types and missing values
`.info()` shows dtype + non-null count per column. Watch for nulls in `business_address` or `country` — missing data changes how we engineer features later.

In [5]:
s1.info()

<class 'pandas.DataFrame'>
RangeIndex: 2206821 entries, 0 to 2206820
Data columns (total 4 columns):
 #   Column            Dtype
---  ------            -----
 0   entity_id         str  
 1   business_name     str  
 2   business_address  str  
 3   country           str  
dtypes: str(4)
memory usage: 67.3 MB


### 1.4 Country distribution
`.value_counts()` counts rows per category. Confirm US/India split in train, and remember: test also has France, which we must NOT hardcode against.

In [6]:
print("Source 1 country counts:")
print(s1["country"].value_counts())
print()
print("Source 2 country counts:")
print(s2["country"].value_counts())


Source 1 country counts:
country
US       1323633
India     883188
Name: count, dtype: int64

Source 2 country counts:
country
US       3016817
India    2017799
Name: count, dtype: int64


### 1.5 Ground truth shape of the problem
`matched_entity_ids` is a comma-separated string, or empty/NaN for singletons.
Let's count: how many Source-1 entities have ZERO matches (singletons) vs. some matches?

In [7]:
gt["is_singleton"] = gt["matched_entity_ids"].isna() | (gt["matched_entity_ids"] == "")
print("Singleton count:", gt["is_singleton"].sum())
print("Non-singleton count:", (~gt["is_singleton"]).sum())
print("Singleton fraction:", gt["is_singleton"].mean())


Singleton count: 123247
Non-singleton count: 2083574
Singleton fraction: 0.05584820880352326


### 1.6 How many matches per non-singleton entity?
We split the comma-separated string into a list and count its length using `.apply()` — runs a function on every row.

In [8]:
def count_matches(x):
    if pd.isna(x) or x == "":
        return 0
    return len(x.split(","))

gt["num_matches"] = gt["matched_entity_ids"].apply(count_matches)
print(gt["num_matches"].value_counts().sort_index())
print()
print("Max matches for a single entity:", gt["num_matches"].max())


num_matches
0     123247
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64

Max matches for a single entity: 11


**Pause here.** Run everything above and look at the printed output before moving on — 
we'll discuss what these numbers mean for our strategy before touching Step 2.

## Step 2: Understand the Noise + Clean the Data

Goal: (a) look at real match examples to catalogue noise patterns, (b) quantify how severe
each pattern is, (c) build a first-pass cleaning/normalization function and apply it to all
three sources so later steps (blocking, features) work on consistent text.


### 2.1 Build ID lookups

`.set_index()` turns a column into the DataFrame's index so `.loc[]` lookups by ID are fast
(important once we do this millions of times). We reuse `s1_lookup`, `s2_lookup`, `s3_lookup`,
and `lookup_any()` throughout the rest of the notebook.

In [9]:
s1_lookup = s1.set_index("entity_id")
s2_lookup = s2.set_index("entity_id")
s3_lookup = s3.set_index("entity_id")

def lookup_any(eid):
    if eid.startswith("S2-"):
        return s2_lookup.loc[eid]
    elif eid.startswith("S3-"):
        return s3_lookup.loc[eid]
    else:
        raise ValueError(f"Unexpected id prefix: {eid}")


### 2.2 Inspect real matched pairs

Pull a handful of non-singleton Source 1 entities and print their matches side by side,
so we see real noise instead of guessing from the problem statement.

In [10]:
sample_gt = gt[~gt["is_singleton"]].sample(5, random_state=42)

for _, row in sample_gt.iterrows():
    s1_row = s1_lookup.loc[row["source1_entity_id"]]
    print("=" * 80)
    print(f"SOURCE 1: {row['source1_entity_id']}")
    print(f"  name:    {s1_row['business_name']}")
    print(f"  address: {s1_row['business_address']}")
    print(f"  country: {s1_row['country']}")
    print("-" * 80)

    matched_ids = row["matched_entity_ids"].split(",")
    for mid in matched_ids:
        m_row = lookup_any(mid)
        print(f"MATCH: {mid}")
        print(f"  name:    {m_row['business_name']}")
        print(f"  address: {m_row['business_address']}")
        print(f"  country: {m_row['country']}")
    print()


SOURCE 1: S1-777210964
  name:    Davis Family Office
  address: 88 Olive Circle, Lebanon, TN
  country: US
--------------------------------------------------------------------------------
MATCH: S2-12029274
  name:    Davis Family Offie
  address: 88 OLIVE CIR, LEBANON, TN
  country: US
MATCH: S2-407524980
  name:    DAVIS FAMILY OFFICE
  address: OLIVE CIR, LEBANON, TN
  country: US
MATCH: S3-970714334
  name:    Davis Family (Office)
  address: 88 Olive Cir, Lebanon, Tennessee
  country: US
MATCH: S3-712922821
  name:    Davis Family
  address: Lebanon, Tennessee, 88 Olive Cir
  country: US

SOURCE 1: S1-190133982
  name:    MS Consultancy Corp
  address: Shymala Appts 1St Floor Flat No. 4 Opp Ratna Hospital Sb Road In Haveli, Pune, Maharashtra
  country: India
--------------------------------------------------------------------------------
MATCH: S2-523219173
  name:    MS [Consultancy]
  address: SHYMALA APPTS 1-1ST FLOOR FLAT NO. 4 OPP RATNA HOSPITAL SB ROAD IN HAVELI, PUNE, Maha

### 2.3 Noise catalogue (from inspection)

**Name noise:**
- Case only: `Davis Family Office` → `DAVIS FAMILY OFFICE`
- Typos: `Office` → `Offie`, `MS` → `M5`
- Punctuation/brackets: `MS Consultancy Corp` → `MS [Consultancy]`
- Legal suffix variants: `Pvt. Ltd.` ↔ `Pvt. Limited`
- Dropped words: `Davis Family Office` → `Davis Family`
- Substituted-but-related words: `MS Consultancy Corp` → `MS Corp Services`
- **Completely unrelated strings (DBA / trade names)**: `Team Air Pvt. Ltd.` → `Mirasol` —
  zero characters in common, still a true match. No name-similarity feature will ever catch this;
  address has to carry the signal here.

**Address noise:**
- Abbreviation swaps: `Circle` ↔ `Cir`, `Tennessee` ↔ `TN`
- Component reordering: `88 Olive Circle, Lebanon, TN` vs `Lebanon, Tennessee, 88 Olive Cir`
- Missing components: house number dropped entirely
- Script/transliteration: Telugu script vs `Telangana` vs `Andhra Pradesh` for the same region
- Junk symbols: `##19821 WHEELWRIGHT DR`

**Implication:** name and address similarity must stay as *separate* features (never collapsed
into one score), since a true match can have near-zero name similarity but near-one address
similarity, or vice versa.


### 2.4 How common is "name is basically useless"?

We saw one dramatic example (`Mirasol`). Before designing blocking around it, we need to know:
is this a rare outlier, or a real chunk of the data? We measure **word overlap** (Jaccard
similarity on the set of words) between Source 1 names and their true matches, across a
larger random sample.

In [11]:
def word_overlap(name1, name2):
    w1 = set(str(name1).lower().split())
    w2 = set(str(name2).lower().split())
    if not w1 or not w2:
        return 0
    return len(w1 & w2) / len(w1 | w2)

sample_gt2 = gt[~gt["is_singleton"]].sample(2000, random_state=1)

zero_overlap_count = 0
total_pairs = 0
overlaps = []

for _, row in sample_gt2.iterrows():
    s1_name = s1_lookup.loc[row["source1_entity_id"], "business_name"]
    matched_ids = row["matched_entity_ids"].split(",")
    for mid in matched_ids:
        m_name = lookup_any(mid)["business_name"]
        overlap = word_overlap(s1_name, m_name)
        overlaps.append(overlap)
        total_pairs += 1
        if overlap == 0:
            zero_overlap_count += 1

print(f"Total match pairs checked: {total_pairs}")
print(f"Pairs with ZERO word overlap in name: {zero_overlap_count}")
print(f"Fraction with zero overlap: {zero_overlap_count / total_pairs:.4f}")

import pandas as pd
print()
print("Overlap score distribution:")
print(pd.Series(overlaps).describe())


Total match pairs checked: 7403
Pairs with ZERO word overlap in name: 1164
Fraction with zero overlap: 0.1572

Overlap score distribution:
count    7403.000000
mean        0.562595
std         0.322453
min         0.000000
25%         0.333333
50%         0.600000
75%         0.750000
max         1.000000
dtype: float64


**Read this before moving on:** if the zero-overlap fraction is small (a few percent), name
similarity is still our primary signal and address just rescues the occasional edge case.
If it's large (double digits), address similarity needs to be a first-class signal in blocking
too, not just a tiebreaker fed to the model after name-based blocking already discarded the
true candidate. Paste your numbers back before we design blocking in Step 3.


### 2.5 Cleaning / normalization strategy

Goal: reduce *superficial* noise (case, punctuation, common abbreviations) so that similarity
features in Step 4 aren't fooled by formatting differences that don't reflect a real business
difference. This does **not** solve the DBA/trade-name problem (`Mirasol`) — no amount of
normalization fixes a genuinely different string. That's a matching-model problem, not a
cleaning problem.

What we normalize:
- Lowercase everything
- Strip punctuation (but keep spaces as word separators)
- Collapse repeated whitespace
- Expand common street-type abbreviations (`rd` → `road`, `st` → `street`, etc.)
- Expand common legal-suffix abbreviations in names (`corp` → `corporation`, `ltd` → `limited`,
  `pvt` → `private`, `inc` → `incorporated`, `&` → `and`)
- Strip leading junk characters like `#`

What we deliberately do NOT attempt here (flagging honestly, not hiding it):
- Transliterating non-Latin script (e.g. Telugu) into Latin script — this is a hard NLP problem
  on its own and the rules disallow external APIs/services to do it for us
- Fixing typos automatically (that's what fuzzy/edit-distance similarity in Step 4 is for)
- Resolving DBA/trade names (that's what the *model*, learning from address signal, is for)


In [12]:
import re

# Street-type abbreviations -> canonical form. Add to this as you discover more from the data.
STREET_ABBREV = {
    r"\brd\b": "road",
    r"\bst\b": "street",
    r"\bave\b": "avenue",
    r"\bdr\b": "drive",
    r"\bcir\b": "circle",
    r"\bblvd\b": "boulevard",
    r"\bln\b": "lane",
    r"\bapt\b": "apartment",
    r"\bbldg\b": "building",
    r"\bfl\b": "floor",
}

# Legal-suffix / business-name abbreviations -> canonical form.
NAME_ABBREV = {
    r"\bcorp\b": "corporation",
    r"\bltd\b": "limited",
    r"\bpvt\b": "private",
    r"\binc\b": "incorporated",
    r"\bllc\b": "llc",     # already a standard token, kept for clarity/no-op
    r"&": " and ",
}

def clean_text(text, abbrev_map):
    if text is None or (isinstance(text, float)):
        return ""
    t = str(text).lower()
    t = t.replace("#", " ")           # strip junk symbols like ##19821
    t = re.sub(r"[^\w\s]", " ", t)    # strip punctuation, keep word chars + spaces
    for pattern, replacement in abbrev_map.items():
        t = re.sub(pattern, replacement, t)
    t = re.sub(r"\s+", " ", t).strip() # collapse repeated whitespace
    return t

def clean_name(name):
    return clean_text(name, NAME_ABBREV)

def clean_address(address):
    return clean_text(address, STREET_ABBREV)


### 2.6 Apply cleaning to all three sources

This adds `business_name_clean` and `business_address_clean` columns to `s1`, `s2`, `s3`.
On ~12M combined rows this can take a couple of minutes with `.apply()` — that's expected;
we're trading a one-time cost now for consistent, reusable clean columns later.

In [13]:
for df, label in [(s1, "Source 1"), (s2, "Source 2"), (s3, "Source 3")]:
    df["business_name_clean"] = df["business_name"].apply(clean_name)
    df["business_address_clean"] = df["business_address"].apply(clean_address)
    print(f"{label} cleaned.")

s1[["business_name", "business_name_clean", "business_address", "business_address_clean"]].head(10)


Source 1 cleaned.
Source 2 cleaned.
Source 3 cleaned.


,business_name,business_name_clean,business_address,business_address_clean
0,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc
1,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok
2,B+ Retail Inc,b retail incorporated,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az
3,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md
4,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal
5,Custom Wealth Services LLC,custom wealth services llc,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...
7,Nexus Anchor Rain,nexus anchor rain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn
8,Moore Bitwise Inc,moore bitwise incorporated,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in
9,Dermatology Green Medicine,dermatology green medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...


### 2.7 Save cleaned data (avoid recomputing every run)

Cleaning ~12M rows takes real time — save the result once so future notebook runs (and later
steps) can just reload it instead of re-cleaning from scratch. Parquet is compact and fast for
this compared to re-parsing TSV + re-cleaning every time.

In [ ]:
import os
os.makedirs("dataset/clean", exist_ok=True)

s1.to_parquet("dataset/clean/s1_clean.parquet")
s2.to_parquet("dataset/clean/s2_clean.parquet")
s3.to_parquet("dataset/clean/s3_clean.parquet")

print("Saved cleaned sources to dataset/clean/")


**If `to_parquet` errors**, you likely need the `pyarrow` package:
run `pip install pyarrow` in your terminal (same environment as your kernel), then re-run the
cell above.

**Pause here.** Run Step 2 top to bottom, paste back:
1. The word-overlap numbers from 2.4
2. A few rows of the cleaned output from 2.6 (does `Cir` become `circle`, does `Corp` become
   `corporation`, etc.?)

Then we'll move to Step 3: designing the actual blocking strategy, informed by what we now
know about how often name alone is enough vs. not.


## Step 3: Blocking / Candidate Generation (TODO)

Goal: for each Source 1 entity, cheaply narrow down a small set of plausible Source 2 / Source 3
candidates. This sets our recall ceiling — nothing later can recover a match blocking misses.


## Step 4: Feature Engineering (TODO)

Goal: turn each (Source1, candidate) pair into numeric similarity features
(Jaccard, Levenshtein, TF-IDF cosine, etc. on name and address).


## Step 5: Model Training (TODO)

Goal: train a binary classifier (match / not match) on labeled pairs built from
`train_ground_truth.tsv` + our candidate pairs.


## Step 6: Validation Split + F0.5 Evaluation (TODO)

Goal: since test has no ground truth, we carve out our own validation slice of train
to self-score before submitting. Implement the F0.5 formula from the problem statement.


## Step 7: Generate Submission Files (TODO)

Goal: produce `output/matching_results.tsv` and `output/candidate_pairs.tsv` for the test set,
then validate with `utils/validate_submission.py`.
